# Detector de Emociones Faciales
**Narciso Beras - 24-EISN-2-026 - Inteligencia Artificial**

---

### Antes de comenzar:
1. Ve a **Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion**
2. Selecciona **GPU T4** y guarda
3. Ejecuta las celdas en orden de arriba hacia abajo


## Celda 1 - Montar Google Drive
> Se te pedira permiso para acceder a tu Drive. Acepta y el modelo se guardara automaticamente ahi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUTA_MODELO = '/content/drive/MyDrive/models'
os.makedirs(RUTA_MODELO, exist_ok=True)

print(f'Drive montado correctamente')
print(f'Los modelos se guardaran en: {RUTA_MODELO}')

## Celda 2 - Instalar dependencias

In [ ]:
!pip install -q timm datasets tqdm
print('Dependencias instaladas correctamente')

## Celda 3 - Verificar GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'GPU detectada: {torch.cuda.get_device_name(0)}')
    print(f'Memoria disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No se detecto GPU. Ve a Entorno de ejecucion -> Cambiar tipo -> GPU T4')

## Celda 4 - Configuracion

In [ ]:
CLASES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
NUM_CLASES = 7
EPOCAS = 10
LR = 1e-3
BS = 64
SIZE = 48
MEDIA = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Dispositivo : {DISPOSITIVO}')
print(f'Epocas      : {EPOCAS}')
print(f'Batch size  : {BS}')

## Celda 5 - Transformaciones e imports

In [ ]:
import timm
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from tqdm.notebook import tqdm

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(SIZE),
    transforms.Resize([SIZE, SIZE]),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA, STD),
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize([SIZE, SIZE]),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA, STD),
])

print('Transformaciones listas')

## Celda 6 - Dataset y Modelo

In [ ]:
class FERDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        imagen = self.data[idx]['image']
        etiqueta = self.data[idx]['label']
        if self.transform:
            imagen = self.transform(imagen)
        return imagen, etiqueta


class EmotionModel(nn.Module):
    def __init__(self, num_clases):
        super().__init__()
        self.model = timm.create_model('resnet18', pretrained=True)
        for param in self.model.parameters():
            param.requires_grad = False
        self.model.fc = nn.Linear(self.model.fc.in_features, num_clases)

    def forward(self, x):
        return self.model(x)


def accuracy(preds, labels):
    _, pred_clases = torch.max(preds, dim=1)
    return (pred_clases == labels).sum().item() / labels.size(0)


print('Clases definidas correctamente')

## Celda 7 - Descargar dataset FER-2013
> La descarga puede tardar 1-2 minutos (~300 MB)

In [ ]:
print('Descargando FER-2013...')
dataset = load_dataset('trpakov/fer-facial-expression-recognition', 'base')
train_data = dataset['train']
test_data = dataset['validation']

train_dataset = FERDataset(train_data, transform=train_transform)
test_dataset = FERDataset(test_data, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BS, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BS*2, shuffle=False, num_workers=2)

print(f'Dataset cargado')
print(f'Entrenamiento : {len(train_dataset):,} imagenes')
print(f'Validacion    : {len(test_dataset):,} imagenes')

## Celda 8 - Entrenamiento
> El modelo se guardara automaticamente en tu Google Drive en la carpeta **models/**


In [ ]:
modelo = EmotionModel(NUM_CLASES).to(DISPOSITIVO)
perdida_fn = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=LR)

mejor_accuracy = 0.0
historial = []
RUTA_GUARDADO = f'{RUTA_MODELO}/modelo_emociones.pth'

for epoca in range(EPOCAS):
    modelo.train()
    perdida_train = acc_train = 0

    for imagenes, etiquetas in tqdm(train_loader, desc=f'Epoca {epoca+1}/{EPOCAS} Train'):
        imagenes, etiquetas = imagenes.to(DISPOSITIVO), etiquetas.to(DISPOSITIVO)
        optimizador.zero_grad()
        preds = modelo(imagenes)
        perdida = perdida_fn(preds, etiquetas)
        perdida.backward()
        optimizador.step()
        perdida_train += perdida.item()
        acc_train += accuracy(preds, etiquetas)

    perdida_train /= len(train_loader)
    acc_train /= len(train_loader)

    modelo.eval()
    perdida_test = acc_test = 0

    for imagenes, etiquetas in tqdm(test_loader, desc=f'Epoca {epoca+1}/{EPOCAS} Test'):
        imagenes, etiquetas = imagenes.to(DISPOSITIVO), etiquetas.to(DISPOSITIVO)
        with torch.no_grad():
            preds = modelo(imagenes)
            perdida = perdida_fn(preds, etiquetas)
            perdida_test += perdida.item()
            acc_test += accuracy(preds, etiquetas)

    perdida_test /= len(test_loader)
    acc_test /= len(test_loader)

    historial.append({'epoca': epoca+1, 'acc_train': acc_train, 'acc_test': acc_test})

    guardado = ''
    if acc_test > mejor_accuracy:
        mejor_accuracy = acc_test
        torch.save(modelo.state_dict(), RUTA_GUARDADO)
        guardado = f'  [Guardado en Drive]'

    print(f'Epoca {epoca+1:02d} | Train loss={perdida_train:.4f} acc={acc_train:.4f} | Test loss={perdida_test:.4f} acc={acc_test:.4f}{guardado}')

print(f'Entrenamiento completo. Mejor accuracy: {mejor_accuracy:.4f}')
print(f'Modelo guardado en: {RUTA_GUARDADO}')

## Celda 9 - Grafica de resultados

In [ ]:
import matplotlib.pyplot as plt

epocas = [h['epoca'] for h in historial]
acc_tr = [h['acc_train'] for h in historial]
acc_te = [h['acc_test'] for h in historial]

plt.figure(figsize=(8, 4))
plt.plot(epocas, acc_tr, marker='o', label='Train accuracy', color='#1D9E75')
plt.plot(epocas, acc_te, marker='s', label='Test accuracy', color='#378ADD')
plt.title('Accuracy por epoca')
plt.xlabel('Epoca')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Celda 10 - Verificar archivo en Drive

In [ ]:
import os

if os.path.exists(RUTA_GUARDADO):
    size_mb = os.path.getsize(RUTA_GUARDADO) / 1e6
    print(f'Modelo encontrado en Drive')
    print(f'Ruta  : {RUTA_GUARDADO}')
    print(f'Tamano: {size_mb:.1f} MB')
else:
    print('No se encontro el modelo. Verifica que el entrenamiento haya completado.')